# Apache Spark avanzado: Optimización y técnicas de performance

**Sesión 26:** Optimización de Spark I — Broadcast Joins y Particionamiento (parte 2 de 2)

**Dataset:** NYC Taxi Trip Data (Yellow Taxi - 2024)  


### Reinicialización de la SparkSession

La sesión de Spark (SparkSession) **no persiste** entre clases, aunque los datos cargados en HDFS durante la Sesión 25 **sí siguen disponibles**. Antes de continuar con el Paso 4, se reinicializa Spark con la misma configuración de la Sesión 25 y se vuelve a leer el dataset ya cargado en HDFS, sin necesidad de descargarlo ni subirlo nuevamente.

In [1]:
# Reinicialización de SparkSession y recarga del dataset ya disponible en HDFS
# (la SparkSession no persiste entre clases; los datos en HDFS sí)

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,           # Para acceder a columnas
    avg,           # Promedios
    sum as spark_sum,  # Suma (renombrado para evitar conflicto con sum() de Python)
    count,         # Conteo
    when,          # Condicionales
    to_date,       # Conversión a fecha
    concat,        # Concatenación de strings
    lit,           # Valores literales
    broadcast      # Hint de broadcast join
)
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel  # Para niveles de persistencia
import time
import subprocess

# Crear SparkSession con la misma configuración usada en la Sesión 25
spark = SparkSession.builder \
    .master("yarn") \
    .appName("NYC Taxi Analysis") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Verificar configuración
print("Spark Version:", spark.version)
print("Master:", spark.sparkContext.master)
print("App ID:", spark.sparkContext.applicationId)

# Ruta base en HDFS (ya creada y poblada en la Sesión 25)
hdfs_base_path = "hdfs://hadoop-virtualbox:9000/datasets"

# Releer el dataset completo de 2024, ya cargado en HDFS — no requiere descarga ni carga nuevamente
df_taxi = spark.read.parquet(f"{hdfs_base_path}/yellow_tripdata_2024-*.parquet")

print(f"\nDataset recargado desde HDFS")
print(f"Particiones: {df_taxi.rdd.getNumPartitions()}")
print(f"Número de columnas: {len(df_taxi.columns)}")


Spark Version: 3.5.7
Master: yarn
App ID: application_1790802456158_0002



Dataset recargado desde HDFS
Particiones: 6
Número de columnas: 19


---

## Sección 4: Tipos de joins

### Joins en Spark

#### Tipos de joins físicos

Spark implementa principalmente dos estrategias de join:

##### **Sort Merge Join** (estrategia por defecto con datos grandes)
```
DataFrame A          DataFrame B
    |                    |
  Shuffle             Shuffle
    |                    |
    +----- JOIN ---------+
```
- Ambos DataFrames se redistribuyen (shuffle) y ordenan por la clave del join
- **Costoso:** Movimiento de datos entre nodos + ordenamiento
- **Usa mucha red:** Shuffle write/read
- **Escalable:** Funciona con DataFrames muy grandes

##### **Broadcast Hash Join**
```
DataFrame A (grande)    DataFrame B (pequeño)
    |                          |
    |                      [Broadcast]
    |                          |
    +-------- JOIN ◄-----------+
       (sin shuffle)
```
- El DataFrame **pequeño** se copia a todos los nodos
- **Rápido:** No hay shuffle del DataFrame grande
- **Limitación:** El DataFrame pequeño debe caber en memoria
- **Umbral:** Por defecto 10MiB (`spark.sql.autoBroadcastJoinThreshold`)

### ¿Cuándo usar cada uno?

| Escenario | Join Recomendado |
|-----------|------------------|
| Tabla grande + tabla pequeña (menos de 10MiB) | Broadcast Hash Join |
| Dos tablas grandes | Sort Merge Join |
| Join con tabla de dimensiones | Broadcast Hash Join |
| Join con múltiples filtros | Considerar filtrar antes del join |

### Paso 4.1: Preparar datasets para join

Se van a crear dos datasets:
1. **Dataset grande:** Viajes del año completo 2024 (con filtros básicos de calidad)
2. **Dataset pequeño:** Información de zonas (location IDs)

In [2]:
print("Preparando datasets para joins...\n")

# Dataset 1: Viajes completos (grande)
df_trips = df_taxi.select(
    "VendorID",
    "tpep_pickup_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount"
).filter(
    (col("trip_distance") > 0) & 
    (col("fare_amount") > 0)
)

trips_count = df_trips.count()
print(f"Dataset de viajes (grande): {trips_count:,} registros")

# Dataset 2: Información de zonas (pequeño - simulado)
# En un caso real, esto vendría de un archivo de referencia
# Para este ejemplo, crearemos zonas sintéticas basadas en los IDs únicos
from pyspark.sql.functions import concat, lit

df_zones = df_taxi.select("PULocationID").distinct() \
    .withColumnRenamed("PULocationID", "LocationID") \
    .withColumn("zone_name", concat(lit("Zone_"), col("LocationID"))) \
    .withColumn("borough", 
        when(col("LocationID") <= 100, "Manhattan")
        .when(col("LocationID") <= 150, "Brooklyn")
        .when(col("LocationID") <= 200, "Queens")
        .otherwise("Other")
    )

zones_count = df_zones.count()
print(f"Dataset de zonas (pequeño): {zones_count:,} registros")

print("\nDatasets preparados para join")

Preparando datasets para joins...



Dataset de viajes (grande): 39,706,652 registros


[Stage 4:=======================================>                   (4 + 2) / 6]

Dataset de zonas (pequeño): 263 registros

Datasets preparados para join


### Paso 4.2: Sort Merge Join (estrategia por defecto con datos grandes)

In [3]:
print("SORT MERGE JOIN (estrategia por defecto con datos grandes)\n")
print("=" * 70)

# Deshabilitar broadcast join temporalmente para forzar shuffle
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("Broadcast join deshabilitado (threshold = -1)")
print("   Esto forzará un Sort Merge Join\n")

# Realizar join
print("Ejecutando join con shuffle...")
start_time = time.time()

df_joined_shuffle = df_trips.join(
    df_zones,
    df_trips["PULocationID"] == df_zones["LocationID"],
    "inner"
).select(
    df_trips["*"],
    df_zones["zone_name"],
    df_zones["borough"]
)

# Ejecutar acción para medir tiempo
result_count_shuffle = df_joined_shuffle.count()
shuffle_time = time.time() - start_time

print(f"\nJoin completado")
print(f"Tiempo: {shuffle_time:.2f}s")
print(f"Registros resultantes: {result_count_shuffle:,}")

SORT MERGE JOIN (estrategia por defecto con datos grandes)

Broadcast join deshabilitado (threshold = -1)
   Esto forzará un Sort Merge Join

Ejecutando join con shuffle...


[Stage 14:================================================>         (5 + 1) / 6]


Join completado
Tiempo: 7.82s
Registros resultantes: 39,706,652


### Analizar el plan físico con .explain()

In [4]:
print("Plan de ejecución del Sort Merge Join:\n")
print("=" * 70)
df_joined_shuffle.explain(mode="formatted")
print("=" * 70)

print("\nQué buscar en el plan:")
print("   • 'SortMergeJoin': estrategia de join por defecto con datos grandes")
print("   • 'Exchange': redistribución de datos entre nodos (shuffle)")
print("   • 'Sort': ordenamiento previo al merge en cada partición")
print("   • Ambos DataFrames tienen 'Exchange' + 'Sort' antes del join")
print("\nShuffle significa:")
print("   1. Datos se redistribuyen entre nodos")
print("   2. Mucho I/O de red")
print("   3. Se escriben a disco temporalmente (shuffle write)")
print("   4. Se leen de disco (shuffle read)")

Plan de ejecución del Sort Merge Join:

== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- SortMergeJoin Inner (11)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan parquet  (1)
      +- Sort (10)
         +- HashAggregate (9)
            +- Exchange (8)
               +- HashAggregate (7)
                  +- Filter (6)
                     +- Scan parquet  (5)


(1) Scan parquet 
Output [6]: [VendorID#0, tpep_pickup_datetime#1, trip_distance#4, PULocationID#7, DOLocationID#8, fare_amount#10]
Batched: true
Location: InMemoryFileIndex [hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-01.parquet, ... 11 entries]
PushedFilters: [IsNotNull(trip_distance), IsNotNull(fare_amount), GreaterThan(trip_distance,0.0), GreaterThan(fare_amount,0.0), IsNotNull(PULocationID)]
ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,trip_distance:double,PULocationID:int,DOLocationID:int,fare_amount:double>

(2) Filter
Input 

### Paso 4.3: Broadcast Hash Join (optimizado)

In [5]:
print("BROADCAST HASH JOIN (optimizado)\n")
print("=" * 70)

# Habilitar broadcast join (configurar threshold a 10MiB)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10MiB
print("Broadcast join habilitado (threshold = 10MiB)")
print("   DataFrame pequeño se copiará a todos los nodos\n")

# Realizar join
print("Ejecutando join con broadcast...")
start_time = time.time()

df_joined_broadcast = df_trips.join(
    df_zones,
    df_trips["PULocationID"] == df_zones["LocationID"],
    "inner"
).select(
    df_trips["*"],
    df_zones["zone_name"],
    df_zones["borough"]
)

# Ejecutar acción para medir tiempo
result_count_broadcast = df_joined_broadcast.count()
broadcast_time = time.time() - start_time

print(f"\nJoin completado")
print(f"Tiempo: {broadcast_time:.2f}s")
print(f"Registros resultantes: {result_count_broadcast:,}")

BROADCAST HASH JOIN (optimizado)

Broadcast join habilitado (threshold = 10MiB)
   DataFrame pequeño se copiará a todos los nodos

Ejecutando join con broadcast...


[Stage 24:================================================>         (5 + 1) / 6]


Join completado
Tiempo: 4.41s
Registros resultantes: 39,706,652


### Analizar el plan del Broadcast Join

> **Nota sobre AQE:** Con `spark.sql.adaptive.enabled=true`, `.explain()` muestra el plan **inicial**, no el plan final que AQE ejecutó. Por eso el plan puede mostrar `SortMergeJoin` aunque el join se haya ejecutado como `BroadcastHashJoin`. El indicador `isFinalPlan=false` en la salida confirma esto. El plan final real es visible en la Spark UI una vez completada la acción.

In [8]:
print("Plan de ejecución (plan inicial AQE — isFinalPlan=false):\n")
print("=" * 70)
df_joined_broadcast.explain(mode="formatted")
print("=" * 70)

print("\nObservación importante — AQE y el plan inicial:")
print("   • El plan muestra 'SortMergeJoin' porque isFinalPlan=false")
print("   • Con AQE activo, .explain() captura el plan INICIAL, no el final")
print("   • AQE puede re-optimizar el join a BroadcastHashJoin en tiempo de ejecución")
print("   • El plan final real solo es visible en Spark UI (tras completar la acción)")
print("\nBroadcast Hash Join (plan final, cuando AQE lo aplica):")
print("   1. DataFrame pequeño se copia a cada nodo")
print("   2. DataFrame grande NO se mueve (sin shuffle)")
print("   3. Join se hace localmente en cada partición")
print("   4. El Paso 4.6 (con broadcast() explicito) fuerza el plan desde el inicio")

Plan de ejecución (plan inicial AQE — isFinalPlan=false):

== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- SortMergeJoin Inner (11)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan parquet  (1)
      +- Sort (10)
         +- HashAggregate (9)
            +- Exchange (8)
               +- HashAggregate (7)
                  +- Filter (6)
                     +- Scan parquet  (5)


(1) Scan parquet 
Output [6]: [VendorID#0, tpep_pickup_datetime#1, trip_distance#4, PULocationID#7, DOLocationID#8, fare_amount#10]
Batched: true
Location: InMemoryFileIndex [hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-01.parquet, ... 11 entries]
PushedFilters: [IsNotNull(trip_distance), IsNotNull(fare_amount), GreaterThan(trip_distance,0.0), GreaterThan(fare_amount,0.0), IsNotNull(PULocationID)]
ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,trip_distance:double,PULocationID:int,DOLocationID:int,fare_amount:double>

### Paso 4.4: Comparar Performance

In [7]:
print("\n" + "=" * 70)
print("COMPARACIÓN DE PERFORMANCE")
print("=" * 70)

print(f"Sort Merge Join:      {shuffle_time:.2f}s")
print(f"Broadcast Hash Join:  {broadcast_time:.2f}s")
print(f"\n⚡ Speedup: {shuffle_time/broadcast_time:.2f}x más rápido con broadcast")

print("\nExplicación de la diferencia:")
print("   • Sort Merge Join: Mueve AMBOS DataFrames por la red (shuffle + ordenamiento)")
print("   • Broadcast Join: Solo copia el DataFrame pequeño UNA vez")
print("   • Eliminación de shuffle = menos I/O = más velocidad")

print("\nLimitaciones del Broadcast Join:")
print("   • DataFrame pequeño debe caber en memoria de cada executor")
print("   • Si es muy grande, puede causar OutOfMemoryError")
print("   • Típicamente útil para dimensiones inferiores a 100MB")


COMPARACIÓN DE PERFORMANCE
Sort Merge Join:      7.82s
Broadcast Hash Join:  4.41s

⚡ Speedup: 1.77x más rápido con broadcast

Explicación de la diferencia:
   • Sort Merge Join: Mueve AMBOS DataFrames por la red (shuffle + ordenamiento)
   • Broadcast Join: Solo copia el DataFrame pequeño UNA vez
   • Eliminación de shuffle = menos I/O = más velocidad

Limitaciones del Broadcast Join:
   • DataFrame pequeño debe caber en memoria de cada executor
   • Si es muy grande, puede causar OutOfMemoryError
   • Típicamente útil para dimensiones inferiores a 100MB


### Paso 4.5: Analizar Shuffle en Spark UI

**Instrucciones:**
1. Acceder a la **Spark UI** a través de YARN
2. Ir a la pestaña **"Stages"**
3. Buscar el stage del Sort Merge Join
4. Buscar el stage del Broadcast Join

**Métricas a comparar:**
- **Shuffle Write:** Datos escritos durante shuffle
- **Shuffle Read:** Datos leídos durante shuffle
- **Duration:** Tiempo total del stage

El Broadcast Join debe tener shuffle = 0 o muy bajo.

### Paso 4.6: Forzar Broadcast manualmente

In [9]:
# broadcast ya fue importado en la celda inicial; esta línea es solo un recordatorio
# from pyspark.sql.functions import broadcast

print("Forzar broadcast manualmente con broadcast()\n")
print("=" * 70)

# Usar la función broadcast() para forzar el hint
print("Útil cuando:")
print("   • Se sabe que un DataFrame es pequeño pero Spark no lo detecta")
print("   • Se quiere garantizar broadcast sin cambiar configuración\n")

# Ejemplo con hint explícito
df_joined_forced = df_trips.join(
    broadcast(df_zones),  # ← Hint explícito de broadcast
    df_trips["PULocationID"] == df_zones["LocationID"],
    "inner"
)

print("Join configurado con broadcast() explícito")
print("\nPlan de ejecución:")
df_joined_forced.explain(mode="simple")
print("\nDebería verse 'BroadcastHashJoin' en el plan")

Forzar broadcast manualmente con broadcast()

Útil cuando:
   • Se sabe que un DataFrame es pequeño pero Spark no lo detecta
   • Se quiere garantizar broadcast sin cambiar configuración

Join configurado con broadcast() explícito

Plan de ejecución:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [PULocationID#7], [LocationID#57], Inner, BuildRight, false
   :- Project [VendorID#0, tpep_pickup_datetime#1, PULocationID#7, DOLocationID#8, trip_distance#4, fare_amount#10]
   :  +- Filter ((((isnotnull(trip_distance#4) AND isnotnull(fare_amount#10)) AND (trip_distance#4 > 0.0)) AND (fare_amount#10 > 0.0)) AND isnotnull(PULocationID#7))
   :     +- FileScan parquet [VendorID#0,tpep_pickup_datetime#1,trip_distance#4,PULocationID#7,DOLocationID#8,fare_amount#10] Batched: true, DataFilters: [isnotnull(trip_distance#4), isnotnull(fare_amount#10), (trip_distance#4 > 0.0), (fare_amount#10 ..., Format: Parquet, Location: InMemoryFileIndex(12 paths)[hdfs://hadoop-virtu

### Ejercicio 4: Optimizar un Join lento

**Escenario:** Se tiene el siguiente código que es muy lento:

```python
# DataFrame grande: 10M registros
df_transactions = spark.read.parquet("transactions.parquet")

# DataFrame pequeño: 500 categorías
df_categories = spark.read.parquet("categories.parquet")

# Join lento
df_result = df_transactions.join(
    df_categories,
    df_transactions["category_id"] == df_categories["id"],
    "inner"
)
```

**Tareas:**
1. Identificar por qué es lento
2. Proponer 2 optimizaciones diferentes
3. Escribir el código optimizado

In [ ]:
# Escribir su código aquí

---

## Sección 5: Particionamiento y partition pruning

### ¿Qué es el particionamiento?

**Particionamiento** es organizar datos en subdirectorios basados en columnas específicas:

```
Sin particionamiento:              Con particionamiento:
data/                             data/
  ├── part-0001.parquet           ├── year=2024/
  ├── part-0002.parquet           │   ├── month=01/
  └── part-0003.parquet           │   │   └── data.parquet
                                  │   └── month=02/
                                  │       └── data.parquet
                                  └── year=2025/
                                      └── month=01/
                                          └── data.parquet
```

#### Ventajas del particionamiento

1. **Partition pruning:** Skip particiones irrelevantes
   ```python
   # Sin particionamiento: Lee TODOS los datos
   df.filter(col("date") == "2024-01-15")
   
   # Con particionamiento: Solo lee partición relevante
   # Automáticamente salta otras fechas
   ```

2. **Queries más rápidas:** Menos datos escaneados
3. **Organización lógica:** Fácil de entender y mantener
4. **Paralelismo mejorado:** Cada partición puede procesarse independientemente

#### Consideraciones

**Buenas columnas para particionar:**
- Fechas (year, month, day)
- Categorías con baja cardinalidad (país, región, tipo)
- Columnas usadas frecuentemente en filtros WHERE

**Malas columnas para particionar:**
- IDs únicos (crea demasiadas particiones)
- Columnas con alta cardinalidad
- Columnas que casi nunca se usan en filtros

**Consideración:** Evitar crear más de 10.000 particiones

### Paso 5.1: Preparar datos para particionamiento

Se va a agregar una columna de fecha para particionar.

In [10]:
print("Preparando datos para particionamiento...\n")

# Agregar columna de fecha (sin hora) para particionar
# Se filtran registros con fechas fuera de 2024 (datos erroneos en la fuente)
df_to_partition = df_taxi.withColumn(
    "pickup_date",
    to_date(col("tpep_pickup_datetime"))
).filter(
    (col("pickup_date") >= "2024-01-01") & (col("pickup_date") <= "2024-12-31")
).select(
    "VendorID",
    "tpep_pickup_datetime",
    "pickup_date",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count"
)

# Verificar las fechas unicas
print("Fechas únicas en el dataset (primeras y últimas):")
df_to_partition.select("pickup_date").distinct().orderBy("pickup_date").show(5)

print("Columna pickup_date agregada y datos filtrados a 2024")

Preparando datos para particionamiento...

Fechas únicas en el dataset (primeras y últimas):


+-----------+
|pickup_date|
+-----------+
| 2024-01-01|
| 2024-01-02|
| 2024-01-03|
| 2024-01-04|
| 2024-01-05|
+-----------+
only showing top 5 rows

Columna pickup_date agregada y datos filtrados a 2024


### Paso 5.2: Escribir datos particionados

Se usará `.partitionBy()` para crear particiones por fecha.

In [11]:
print("Escribiendo datos particionados en HDFS...\n")

# Ruta de salida particionada
partitioned_path = f"{hdfs_base_path}/taxi_partitioned"

print(f"Ruta de salida: {partitioned_path}")
print("Particionando por: pickup_date\n")

# Escribir con particionamiento
start_time = time.time()

df_to_partition.write \
    .mode("overwrite") \
    .partitionBy("pickup_date") \
    .parquet(partitioned_path)

write_time = time.time() - start_time

print(f"Escritura completa en {write_time:.2f}s")
print("\nCada fecha tendrá su propio subdirectorio")

Escribiendo datos particionados en HDFS...

Ruta de salida: hdfs://hadoop-virtualbox:9000/datasets/taxi_partitioned
Particionando por: pickup_date



Escritura completa en 37.86s

Cada fecha tendrá su propio subdirectorio


### Paso 5.3: Explorar estructura de directorios particionados

In [12]:
print("Explorando estructura de directorios particionados\n")
print("=" * 70)

# Listar directorios de primer nivel (particiones)
result = subprocess.run(
    ["hdfs", "dfs", "-ls", partitioned_path],
    capture_output=True,
    text=True
)
print(result.stdout)

# Mostrar contenido de una partición específica
print("\nContenido de una partición (pickup_date=2024-01-01):")
result = subprocess.run(
    ["hdfs", "dfs", "-ls", f"{partitioned_path}/pickup_date=2024-01-01"],
    capture_output=True,
    text=True
)
print(result.stdout)

print("\nObservaciones:")
print("   • Cada fecha tiene su propio directorio: pickup_date=YYYY-MM-DD/")
print("   • Los archivos .parquet están dentro de cada partición")
print("   • El nombre de la columna está en el path (Hive-style partitioning)")

Explorando estructura de directorios particionados

Found 367 items
-rw-r--r--   2 hadoop supergroup          0 2026-09-30 17:20 hdfs://hadoop-virtualbox:9000/datasets/taxi_partitioned/_SUCCESS
drwxr-xr-x   - hadoop supergroup          0 2026-09-30 17:20 hdfs://hadoop-virtualbox:9000/datasets/taxi_partitioned/pickup_date=2024-01-01
drwxr-xr-x   - hadoop supergroup          0 2026-09-30 17:20 hdfs://hadoop-virtualbox:9000/datasets/taxi_partitioned/pickup_date=2024-01-02
drwxr-xr-x   - hadoop supergroup          0 2026-09-30 17:20 hdfs://hadoop-virtualbox:9000/datasets/taxi_partitioned/pickup_date=2024-01-03
drwxr-xr-x   - hadoop supergroup          0 2026-09-30 17:20 hdfs://hadoop-virtualbox:9000/datasets/taxi_partitioned/pickup_date=2024-01-04
drwxr-xr-x   - hadoop supergroup          0 2026-09-30 17:20 hdfs://hadoop-virtualbox:9000/datasets/taxi_partitioned/pickup_date=2024-01-05
drwxr-xr-x   - hadoop supergroup          0 2026-09-30 17:20 hdfs://hadoop-virtualbox:9000/datasets/taxi_p

### Paso 5.4: Demostrar partition pruning

Se va a comparar lectura con y sin filtros de partición.

In [13]:
print("Demostración Partition Pruning\n")
print("=" * 70)

# Leer datos particionados
df_partitioned = spark.read.parquet(partitioned_path)

# Test 1: Leer TODO el dataset (sin filtro)
print("\nTest 1: Leer TODAS las particiones (sin filtro)")
start = time.time()
count_all = df_partitioned.count()
time_all = time.time() - start
print(f"   Tiempo: {time_all:.2f}s")
print(f"   Registros: {count_all:,}")
print(f"   Lee TODAS las particiones")

# Test 2: Leer solo una partición (con filtro)
print("\nTest 2: Leer UNA partición (pickup_date = '2024-01-15')")
start = time.time()
count_filtered = df_partitioned.filter(col("pickup_date") == "2024-01-15").count()
time_filtered = time.time() - start
print(f"   Tiempo: {time_filtered:.2f}s")
print(f"   Registros: {count_filtered:,}")
print(f"   Lee SOLO 1 partición")

# Comparación
print("\n" + "=" * 70)
speedup = time_all / time_filtered
print(f"Partition pruning es {speedup:.2f} veces más rápido")
# 2024 es año bisiesto: 366 días = 366 particiones de fecha
total_partitions = 366
print(f"\nRazón: Spark saltó {total_partitions - 1} particiones irrelevantes")

Demostración Partition Pruning




Test 1: Leer TODAS las particiones (sin filtro)


   Tiempo: 1.43s
   Registros: 41,169,664
   Lee TODAS las particiones

Test 2: Leer UNA partición (pickup_date = '2024-01-15')
   Tiempo: 0.22s
   Registros: 77,033
   Lee SOLO 1 partición

Partition pruning es 6.57 veces más rápido

Razón: Spark saltó 365 particiones irrelevantes


### Paso 5.5: Ver partition pruning en .explain()

In [14]:
print("Plan de ejecución CON partition pruning:\n")
print("=" * 70)

df_with_filter = df_partitioned.filter(col("pickup_date") == "2024-01-15")
df_with_filter.explain(mode="formatted")

print("\n" + "=" * 70)
print("\nQué buscar en el plan:")
print("   • 'PartitionFilters: [pickup_date = 2024-01-15]'")
print("   • Esto indica que Spark aplicó partition pruning")
print("   • Las particiones que no coinciden se SALTAN completamente")
print("\nSin particionamiento:")
print("   • Tendría que escanear TODOS los archivos")
print("   • Aplicar filtro DESPUÉS de leer los datos")
print("   • Mucho más lento e ineficiente")

Plan de ejecución CON partition pruning:

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [VendorID#273, tpep_pickup_datetime#274, PULocationID#275, DOLocationID#276, trip_distance#277, fare_amount#278, passenger_count#279L, pickup_date#280]
Batched: true
Location: InMemoryFileIndex [hdfs://hadoop-virtualbox:9000/datasets/taxi_partitioned]
PartitionFilters: [isnotnull(pickup_date#280), (pickup_date#280 = 2024-01-15)]
ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,PULocationID:int,DOLocationID:int,trip_distance:double,fare_amount:double,passenger_count:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [8]: [VendorID#273, tpep_pickup_datetime#274, PULocationID#275, DOLocationID#276, trip_distance#277, fare_amount#278, passenger_count#279L, pickup_date#280]




Qué buscar en el plan:
   • 'PartitionFilters: [pickup_date = 2024-01-15]'
   • Esto indica que Spark aplicó partition pruning
   • Las particiones que no coinciden s

### Paso 5.6: Comparar con Dataset no particionado

In [15]:
print("Comparación: Particionado vs no particionado\n")
print("=" * 70)

# Dataset no particionado (ya se tiene)
non_partitioned_path = "hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-*.parquet"

# Leer dataset no particionado con mismo filtro
print("Leyendo dataset NO particionado con filtro de fecha...")
start = time.time()

df_non_partitioned = spark.read.parquet(non_partitioned_path)
df_non_part_filtered = df_non_partitioned.withColumn(
    "pickup_date",
    to_date(col("tpep_pickup_datetime"))
).filter(col("pickup_date") == "2024-01-15")

count_non_part = df_non_part_filtered.count()
time_non_part = time.time() - start

print(f"   Tiempo: {time_non_part:.2f}s")
print(f"   Registros: {count_non_part:,}")

# Comparación final
print("\n" + "=" * 70)
print("RESULTADOS FINALES")
print("=" * 70)
print(f"\nDataset NO particionado: {time_non_part:.2f}s")
print(f"Dataset particionado:    {time_filtered:.2f}s")
print(f"\nMejora: {time_non_part/time_filtered:.2f}x más rápido con particionamiento")

print("\nLecciones aprendidas:")
print("   1. Particionamiento acelera queries con filtros en columnas de partición")
print("   2. Sin particionamiento, se lee TODO el dataset")
print("   3. Usar columnas en WHERE clause como candidatas para particionamiento")
print("   4. Balance: No crear demasiadas particiones (max ~10,000)")

Comparación: Particionado vs no particionado

Leyendo dataset NO particionado con filtro de fecha...
   Tiempo: 1.08s
   Registros: 77,033

RESULTADOS FINALES

Dataset NO particionado: 1.08s
Dataset particionado:    0.22s

Mejora: 4.96x más rápido con particionamiento

Lecciones aprendidas:
   1. Particionamiento acelera queries con filtros en columnas de partición
   2. Sin particionamiento, se lee TODO el dataset
   3. Usar columnas en WHERE clause como candidatas para particionamiento
   4. Balance: No crear demasiadas particiones (max ~10,000)


### Ejercicio 5: Diseñar estrategia de particionamiento

**Escenario:** Se tiene un dataset de ventas con las siguientes columnas:

```python
sales_data:
  - transaction_id (único, 1 millón de registros)
  - customer_id (100.000 clientes únicos)
  - product_id (5.000 productos)
  - category (50 categorías)
  - country (10 países)
  - sale_date (2 años de datos)
  - amount (decimal)
```

**Queries más comunes:**
1. Ventas por país en un rango de fechas
2. Ventas de una categoría específica
3. Total de ventas por mes

**Preguntas:**
1. ¿Qué columna(s) usaría para particionar? ¿Por qué?
2. ¿Usaría una o múltiples columnas de partición?
3. ¿Qué columnas NO usaría? ¿Por qué?

In [ ]:
# Escriba sus respuestas aquí



In [ ]:
# Cerrar la SparkSession
spark.stop()
print("SparkSession cerrada correctamente.")

---

## Mejores prácticas

Antes de ejecutar un job de Spark, preguntarse:

1. **Formato de datos:**
   - [ ] ¿Estoy usando Parquet en vez de CSV?
   - [ ] ¿Necesito todas las columnas o puedo usar `.select()`?

2. **Cache:**
   - [ ] ¿Uso este DataFrame más de una vez?
   - [ ] ¿Cabe en memoria del clúster?
   - [ ] ¿Liberé cache después de usarlo?

3. **Joins:**
   - [ ] ¿Algún DataFrame es pequeño (menor a 10MiB por defecto, o hasta 100MB si se configura manualmente)?
   - [ ] ¿Puedo usar broadcast join?
   - [ ] ¿Revisé el plan con `.explain()`?

4. **Particionamiento:**
   - [ ] ¿Mis queries filtran por alguna columna específica?
   - [ ] ¿Esa columna tiene baja cardinalidad?
   - [ ] ¿Cuántas particiones crearé?

5. **Monitoreo:**
   - [ ] ¿Revisé Spark UI para bottlenecks?
   - [ ] ¿Hay shuffle excesivo?
   - [ ] ¿Algún stage toma mucho más tiempo?

---

### Errores Comunes

#### 🔴 Error 1: "File not found" en HDFS

**Reporte:**
```
AnalysisException: Path does not exist: hdfs://hadoop-virtualbox:9000/datasets/...
```

**Soluciones:**
1. Verificar que HDFS esté corriendo: `jps` (debe mostrar NameNode)
2. Verificar la ruta con: `hdfs dfs -ls hdfs://hadoop-virtualbox:9000/datasets`
3. Verificar configuración del namenode en `core-site.xml`

---

#### 🔴 Error 2: OutOfMemoryError

**Reporte:**
```
java.lang.OutOfMemoryError: Java heap space
```

**Soluciones:**
1. Aumentar `spark.executor.memory`
2. Aumentar `spark.sql.shuffle.partitions`
3. No usar `.collect()` en DataFrames grandes
4. Liberar cache innecesario con `.unpersist()`
5. No hacer broadcast de DataFrames grandes

---

#### 🔴 Error 3: Job muy lento con shuffle

**Reporte:**
- Stage con mucho "Shuffle Write" y "Shuffle Read" en Spark UI
- Job toma mucho más tiempo de lo esperado

**Soluciones:**
1. Verificar si se puede usar broadcast join
2. Filtrar datos ANTES de join
3. Aumentar `spark.sql.shuffle.partitions` si se tienen muchos datos
4. Verificar que `spark.sql.adaptive.enabled=true`

---

#### 🔴 Error 4: Cache no funciona

**Reporte:**
- Se usa `.cache()` pero no se ve mejora
- Storage tab en Spark UI está vacía

**Soluciones:**
1. Verificar que se ejecutó una **acción** después de `.cache()`
   ```python
   df.cache()  # Solo marcar para cache
   df.count()  # Ahora se materializa el cache
   ```
2. Verificar que el DataFrame cabe en memoria
3. Revisar Spark UI → Storage 

---

#### 🔴 Error 5: Demasiadas particiones pequeñas

**Reporte:**
- Miles de archivos pequeños (menor a 1MB) en HDFS
- Lectura muy lenta

**Soluciones:**
1. Usar `.coalesce(N)` antes de escribir
2. Usar `.repartition(N)` si se necesita redistribuir
3. Configurar `spark.sql.files.maxPartitionBytes`

```python
# Antes de escribir
df.coalesce(10).write.parquet("output")
```